# Agentic GitHub Benchmark v2 — issue-driven (`kbench`)

Расширение `benchmark_task_github_skill.ipynb`. Оригинал не изменён.

**Что добавлено:**
1. Расширенный слой GitHub API: issues, comments, trees, compare, pull requests.
2. Новые инструменты агента: `list_files`, `search_code`, `read_file(branch, диапазон строк)`,
   `patch_file`, `delete_file`, `list_issues`, `get_issue`, `create_issue_branch`,
   `comment_issue`, `report_work`, `diff_vs_base`.
3. Новая задача `github_issue_resolution`: агент читает issue, делает изменения
   в отдельной ветке `<slug>/issue-<N>` и комментирует issue ссылкой на ветку.
4. Жёсткий guard: запись разрешена только в ветки с префиксом slug модели.
5. Ретраи и обработка rate limit, обрезка длинных ответов инструментов.
6. Инструмент `report_no_action_needed` + независимый судья: путь зачёта
   для issue, которые честно НЕ требуют правки кода (например, явно просят
   агентов их не трогать) — раньше такие issue были непроходимы в принципе.
7. Джиттер 2-3 сек перед каждым запросом (и к GitHub, и к LLM) + печать
   полного текста исключения и один ретрай на похожие на временные ошибки
   (rate limit/quota/429) — по итогам разбора `PermissionDeniedException`
   у `opus-5`.

**Требования к токену:** fine-grained PAT c `Contents: RW`, **`Issues: RW`**,
`Pull requests: RW` (опционально), `Metadata: R`.


In [1]:
import base64
import functools
import json
import re
import time

import requests

GITHUB_OWNER = "mlaa4ml"
GITHUB_REPO = "durakProekt"
GITHUB_BASE_BRANCH = "main"
GITHUB_API = "https://api.github.com"

try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as e:  # noqa: BLE001
    GITHUB_TOKEN = None
    print(f"Нет GITHUB_TOKEN из Kaggle Secrets ({e}). Add-ons -> Secrets.")

assert GITHUB_TOKEN, "Нужен GITHUB_TOKEN"

GITHUB_HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

REPO_PATH = f"/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
REPO_URL = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}"
MAX_TOOL_OUTPUT = 12000  # символов: длинные ответы режем, чтобы не жечь контекст

## 1. Транспорт: ретраи, rate limit, пагинация

## 2c. Джиттер между запросами + константы для пути "изменения не нужны"

Правки по итогам разбора issue #9 / issue #10 у `opus-5`:

1. Случайная пауза 2–3 сек перед каждым исходящим запросом — и к GitHub, и к LLM.
   Не панацея от `PermissionDeniedException` (похоже на исчерпание бюджета/квоты
   конкретно на `opus-5`, а не на частоту запросов), но дешёвая мера и снижает
   шанс словить обычный rate limit при параллельных прогонах.
2. Полный текст исключения теперь печатается сразу в консоль (не только
   обрезанная строка в `self_report`), и добавлен один ретрай для похожих на
   временные ошибок (rate limit / quota / 429).
3. Константа-маркер и системный промпт судьи для нового пути оценки
   `report_no_action_needed` (вариант A+B из обсуждения).

In [ ]:
import random

# Пауза между исходящими запросами (и к LLM, и к GitHub) — сглаживает всплески
# нагрузки. Диапазон намеренно узкий (2-3 сек), чтобы не раздувать длительность
# прогона: за раунд бывает по 10+ вызовов инструментов.
JITTER_MIN_SEC = 2.0
JITTER_MAX_SEC = 3.0


def _jitter():
    time.sleep(random.uniform(JITTER_MIN_SEC, JITTER_MAX_SEC))


# Строки/подстроки, по которым отличаем "похоже на временную ошибку" (стоит
# один раз повторить попытку) от настоящего отказа в доступе (повтор бесполезен).
RETRYABLE_ERROR_HINTS = ("rate", "quota", "429", "resourceexhausted", "overloaded", "timeout")


def _is_retryable(exc: Exception) -> bool:
    text = f"{type(exc).__name__}: {exc}".lower()
    return any(hint in text for hint in RETRYABLE_ERROR_HINTS)


# Маркер в теле комментария issue, по которому check_issue_solved() отличает
# "агент обоснованно решил не менять код" от обычного отчёта о правке.
NO_ACTION_MARKER = "<!-- NO_ACTION_NEEDED -->"


In [ ]:
def _gh_request(method, path, retries=3, **kwargs):
    """Один запрос к GitHub API с ретраями на 5xx и вторичный rate limit."""
    last = None
    for attempt in range(retries):
        _jitter()
        resp = requests.request(
            method, f"{GITHUB_API}{path}", headers=GITHUB_HEADERS, timeout=30, **kwargs
        )
        last = resp
        if resp.status_code < 500 and resp.status_code != 403:
            return resp
        if resp.status_code == 403 and "rate limit" not in resp.text.lower():
            return resp  # это отказ по правам, ретраить бессмысленно
        time.sleep(2 ** attempt)
    return last


def _gh_paginate(path, params=None, max_pages=10):
    """Собирает все страницы списочного эндпоинта."""
    params = dict(params or {})
    params.setdefault("per_page", 100)
    items, page = [], 1
    while page <= max_pages:
        params["page"] = page
        resp = _gh_request("GET", path, params=params)
        if resp.status_code != 200:
            raise RuntimeError(f"{resp.status_code}: {resp.text[:300]}")
        batch = resp.json()
        if not batch:
            break
        items.extend(batch)
        if len(batch) < params["per_page"]:
            break
        page += 1
    return items


def _clip(text):
    text = str(text)
    if len(text) <= MAX_TOOL_OUTPUT:
        return text
    return text[:MAX_TOOL_OUTPUT] + f"\n...[обрезано, всего {len(text)} символов]"


def _err(label, resp):
    return f"ERROR: {label} failed ({resp.status_code}): {resp.text[:300]}"

## 2. Расширенный слой GitHub: contents, tree, issues, PR

In [3]:
def gh_list_branches():
    try:
        return ", ".join(b["name"] for b in _gh_paginate(f"{REPO_PATH}/branches"))
    except RuntimeError as e:
        return f"ERROR: list_branches failed: {e}"


def gh_branch_sha(branch):
    resp = _gh_request("GET", f"{REPO_PATH}/git/ref/heads/{branch}")
    return resp.json()["object"]["sha"] if resp.status_code == 200 else None


def gh_create_branch(branch_name, base=None):
    base = base or GITHUB_BASE_BRANCH
    if gh_branch_sha(branch_name):
        return f"Ветка {branch_name} уже существует."
    base_sha = gh_branch_sha(base)
    if not base_sha:
        return f"ERROR: базовая ветка {base} не найдена"
    resp = _gh_request(
        "POST", f"{REPO_PATH}/git/refs",
        json={"ref": f"refs/heads/{branch_name}", "sha": base_sha},
    )
    if resp.status_code not in (200, 201):
        return _err("create_branch", resp)
    return f"Ветка {branch_name} создана от {base} ({base_sha[:7]})."


def gh_list_files(branch, subdir=""):
    """Рекурсивный листинг файлов ветки (git/trees?recursive=1)."""
    sha = gh_branch_sha(branch)
    if not sha:
        return f"ERROR: ветка {branch} не найдена"
    resp = _gh_request("GET", f"{REPO_PATH}/git/trees/{sha}", params={"recursive": "1"})
    if resp.status_code != 200:
        return _err("list_files", resp)
    rows = [
        f"{n['path']} ({n.get('size', 0)} B)"
        for n in resp.json().get("tree", [])
        if n["type"] == "blob" and n["path"].startswith(subdir)
    ]
    return _clip("\n".join(rows) or "(пусто)")


def gh_read_file(path, branch, start_line=1, max_lines=None):
    resp = _gh_request("GET", f"{REPO_PATH}/contents/{path}", params={"ref": branch})
    if resp.status_code != 200:
        return f"ERROR: {path} not found on branch {branch} ({resp.status_code})"
    data = resp.json()
    if data.get("encoding") != "base64":
        return f"ERROR: unexpected encoding {data.get('encoding')!r} for {path}"
    text = base64.b64decode(data["content"]).decode("utf-8", errors="replace")
    if start_line > 1 or max_lines:
        lines = text.splitlines()
        end = start_line - 1 + max_lines if max_lines else len(lines)
        text = "\n".join(lines[start_line - 1:end])
    return _clip(text)


def gh_search_code(query, branch, subdir=""):
    """Простой grep по дереву ветки (без Search API — он лагает на свежих коммитах)."""
    listing = gh_list_files(branch, subdir)
    if listing.startswith("ERROR:"):
        return listing
    hits = []
    for row in listing.splitlines():
        path = row.rsplit(" (", 1)[0]
        content = gh_read_file(path, branch)
        if content.startswith("ERROR:"):
            continue
        for i, line in enumerate(content.splitlines(), start=1):
            if query.lower() in line.lower():
                hits.append(f"{path}:{i}: {line.strip()[:200]}")
                if len(hits) >= 100:
                    return _clip("\n".join(hits))
    return _clip("\n".join(hits) or f"Ничего не найдено по '{query}'")


def gh_write_file(path, content, branch, message=None):
    existing = _gh_request("GET", f"{REPO_PATH}/contents/{path}", params={"ref": branch})
    sha = existing.json().get("sha") if existing.status_code == 200 else None
    payload = {
        "message": message or f"{path}: {'update' if sha else 'create'} ({branch})",
        "content": base64.b64encode(content.encode("utf-8")).decode("ascii"),
        "branch": branch,
    }
    if sha:
        payload["sha"] = sha
    resp = _gh_request("PUT", f"{REPO_PATH}/contents/{path}", json=payload)
    if resp.status_code not in (200, 201):
        return _err("write_file", resp)
    commit = resp.json().get("commit", {}).get("sha", "")[:7]
    return f"Записано {len(content)} символов в {path} (ветка {branch}), commit {commit}"


def gh_delete_file(path, branch, message=None):
    existing = _gh_request("GET", f"{REPO_PATH}/contents/{path}", params={"ref": branch})
    if existing.status_code != 200:
        return f"ERROR: {path} нет в ветке {branch}"
    resp = _gh_request("DELETE", f"{REPO_PATH}/contents/{path}", json={
        "message": message or f"{path}: delete ({branch})",
        "sha": existing.json()["sha"],
        "branch": branch,
    })
    return f"Удалён {path} из {branch}" if resp.status_code == 200 else _err("delete_file", resp)


def gh_compare(base, head):
    resp = _gh_request("GET", f"{REPO_PATH}/compare/{base}...{head}")
    if resp.status_code != 200:
        return None
    d = resp.json()
    return {
        "ahead_by": d.get("ahead_by", 0),
        "behind_by": d.get("behind_by", 0),
        "files": [f["filename"] for f in d.get("files", [])],
        "commits": [c["commit"]["message"].splitlines()[0] for c in d.get("commits", [])],
    }


# ---------------- Issues ----------------

# Приоритет задаётся лейблом "priority:p0/p1/p2" (см. раздел 4c). Нет лейбла —
# считаем issue непротриаженным и ставим ему средний ранг (между p1 и p2),
# чтобы такие issues не тонули в конце списка, но и не перебивали явный p0.
PRIORITY_ORDER = ["priority:p0", "priority:p1", "priority:p2"]
PRIORITY_TAGS = {"priority:p0": "P0", "priority:p1": "P1", "priority:p2": "P2"}
UNTRIAGED_RANK = 1.5


def _issue_priority(it):
    """Возвращает (числовой_ранг, тег_для_вывода) по лейблам issue."""
    names = {l["name"] for l in it.get("labels", [])}
    for rank, label in enumerate(PRIORITY_ORDER):
        if label in names:
            return rank, PRIORITY_TAGS[label]
    return UNTRIAGED_RANK, "??"


def gh_list_issues(state="all", labels=None):
    """Все issues репозитория, отсортированные по приоритету (P0 -> P1 -> P2 ->
    непротриаженные). PR отфильтрованы (GitHub отдаёт их тем же эндпоинтом)."""
    params = {"state": state}
    if labels:
        params["labels"] = labels
    try:
        items = [it for it in _gh_paginate(f"{REPO_PATH}/issues", params=params)
                 if "pull_request" not in it]
    except RuntimeError as e:
        return f"ERROR: list_issues failed: {e}"
    items.sort(key=lambda it: _issue_priority(it)[0])
    rows = []
    for it in items:
        _, tag = _issue_priority(it)
        lbl = ",".join(l["name"] for l in it.get("labels", [])) or "-"
        rows.append(
            f"[{tag}] #{it['number']} [{it['state']}] {it['title']} "
            f"(labels: {lbl}; comments: {it.get('comments', 0)})"
        )
    return _clip("\n".join(rows) or "Открытых/закрытых issues нет.")


def gh_get_issue(number):
    resp = _gh_request("GET", f"{REPO_PATH}/issues/{number}")
    if resp.status_code != 200:
        return _err(f"get_issue #{number}", resp)
    it = resp.json()
    out = [
        f"ISSUE #{it['number']}: {it['title']}",
        f"state: {it['state']} | author: {it['user']['login']} | "
        f"labels: {','.join(l['name'] for l in it.get('labels', [])) or '-'}",
        f"url: {it['html_url']}",
        "",
        "--- BODY ---",
        it.get("body") or "(пусто)",
    ]
    try:
        for c in _gh_paginate(f"{REPO_PATH}/issues/{number}/comments"):
            out += ["", f"--- COMMENT by {c['user']['login']} ---", c.get("body") or ""]
    except RuntimeError:
        out.append("\n(комментарии недоступны)")
    return _clip("\n".join(out))


def gh_comment_issue(number, body):
    resp = _gh_request("POST", f"{REPO_PATH}/issues/{number}/comments", json={"body": body})
    if resp.status_code != 201:
        return _err(f"comment_issue #{number}", resp) + "  (нужен scope Issues: RW)"
    return f"Комментарий опубликован: {resp.json()['html_url']}"


def gh_create_pull_request(head, base, title, body):
    resp = _gh_request("POST", f"{REPO_PATH}/pulls", json={
        "head": head, "base": base, "title": title, "body": body,
    })
    if resp.status_code != 201:
        return _err("create_pull_request", resp) + "  (нужен scope Pull requests: RW)"
    return f"PR создан: {resp.json()['html_url']}"


def gh_create_issue(title, body, labels=None):
    """Создаёт новый issue с заданными лейблами (GitHub сам заведёт лейбл priority:*,
    если его ещё нет в репозитории)."""
    payload = {"title": title, "body": body}
    if labels:
        payload["labels"] = list(labels)
    resp = _gh_request("POST", f"{REPO_PATH}/issues", json=payload)
    if resp.status_code != 201:
        return _err("create_issue", resp) + "  (нужен scope Issues: RW)"
    return f"Issue создан: {resp.json()['html_url']}"

## 2b. Патчи issue #16 (`build_toolset`, слой поверх `gh_*` из раздела 2)

Вставлено из `benchmark-task-agentic-github-claude-opus-5-issue16.ipynb` — после раздела 2 (`gh_*`) и до регистрации инструментов (раздел 4). Даёт отдельную точку входа `build_toolset(slug) -> (ctx, tools)` с состоянием рабочей ветки на диске, чтением по умолчанию из рабочей ветки, курсорным чтением больших/минифицированных файлов и исполнением кода. Функции-инструменты из этого слоя нужно регистрировать через `functools.wraps` (см. `with_budget` в разделе 3) — иначе схема инструмента вырождается в `{args, kwargs}` (известный баг проекта).

# Патчи к `benchmark-task-agentic-github-v2bf642(1).ipynb` — issue #16

Автор: агент `anthropic/claude-opus-5@default`, ветка `anthropic-claude-opus-5-default-issue-16`.
Оригинальные ноутбуки не изменены — это отдельный файл с ячейками-заменами.

Ячейки ниже добавляются **после** ячейки со слоем `gh_*` (раздел 2 исходного
ноутбука) и **до** регистрации инструментов агента. Они закрывают 4 пункта issue:

| # | Проблема | Решение в этом ноутбуке |
|---|---|---|
| 1 | Рабочая ветка сбрасывается между раундами (лишний коммит `911c83b`) | `AgentContext` с состоянием на диске (`kbench_agent_state.json`) + guard по slug |
| 2 | `list_files/search_code/read_file` читают `main`, а не рабочую ветку | `build_read_tools(ctx)` — дефолт `from_my_branch=True`, в ответе печатается имя ветки |
| 3 | `MAX_TOOL_OUTPUT` режет ~50 КБ ноутбук | `gh_read_cursor` (курсор `next_start_line`) + `nb_outline` / `nb_cell` / `nb_replace_cell` |
| 4 | Нет исполнения кода и создания issues | `run_python`, `run_repo_file`, `create_issue` |

Требования к токену прежние: `Contents: RW`, `Issues: RW`, `Metadata: R`.

## Fix 1. Рабочая ветка живёт между раундами

In [4]:
# Проблема: выбранная через create_issue_branch(N) ветка хранилась в переменной
# одного прогона. В новом раунде объект пересоздавался, и первый write_file уходил
# в дефолтную ветку slug (лишний коммит 911c83b в anthropic-claude-opus-5-default).
# Решение: контекст агента с состоянием на диске + жёсткий guard по префиксу slug.
import json
import os

AGENT_STATE_PATH = os.environ.get('KBENCH_STATE', '/kaggle/working/kbench_agent_state.json')


def _state_load():
    try:
        with open(AGENT_STATE_PATH, encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return {}


def _state_save(state):
    try:
        with open(AGENT_STATE_PATH, 'w', encoding='utf-8') as f:
            json.dump(state, f, ensure_ascii=False)
    except Exception as e:
        print('state save failed:', e)


class AgentContext:
    '''Контекст одного агента: slug модели + текущая рабочая ветка.

    Ветка пишется и в память, и на диск, поэтому переживает границу раунда
    и рестарт процесса. Записать можно только в ветку с префиксом slug.
    '''

    def __init__(self, slug, base=None):
        self.slug = slug
        self.base = base or GITHUB_BASE_BRANCH
        self.work_branch = _state_load().get(slug, {}).get('work_branch') or slug

    def set_branch(self, branch):
        if not branch.startswith(self.slug):
            raise ValueError('guard: ветка ' + branch + ' вне slug ' + self.slug)
        self.work_branch = branch
        st = _state_load()
        st[self.slug] = {'work_branch': branch}
        _state_save(st)
        return branch

    def resolve(self, from_my_branch=True, branch=None):
        '''Единая точка выбора ветки для всех инструментов.'''
        if branch:
            return branch
        return self.work_branch if from_my_branch else self.base

    def round_header(self, round_no, rounds_total):
        '''Строка, которую раннер обязан добавлять в промпт каждого раунда.'''
        return ('Раунд ' + str(round_no) + '/' + str(rounds_total) +
                '. Текущая рабочая ветка: ' + self.work_branch +
                ' (все записи идут туда, main read-only).')


def make_branch_tools(ctx):
    def create_issue_branch(issue_number: int) -> str:
        '''Создаёт (идемпотентно) ветку <slug>-issue-<N> и делает её текущей на все раунды.'''
        name = ctx.slug + '-issue-' + str(int(issue_number))
        msg = gh_create_branch(name, ctx.base)
        ctx.set_branch(name)
        return msg + ' Текущая рабочая ветка: ' + name

    def use_branch(branch: str) -> str:
        '''Явно переключить рабочую ветку (только внутри своего slug).'''
        try:
            ctx.set_branch(branch)
        except ValueError as e:
            return 'ERROR: ' + str(e)
        return 'Рабочая ветка: ' + ctx.work_branch

    def current_branch() -> str:
        '''Показать текущую рабочую ветку (дешёвая проверка в начале раунда).'''
        return ctx.work_branch

    return create_issue_branch, use_branch, current_branch

## Fix 2. Чтение по умолчанию из рабочей ветки

In [5]:
# Проблема: list_files/search_code/read_file молча читали main, поэтому агент
# не видел собственных коммитов и мог перезаписать свою же работу.
# Решение: дефолт from_my_branch=True + имя ветки в первой строке ответа.

def build_read_tools(ctx):
    def list_files(subdir: str = '', from_my_branch: bool = True) -> str:
        '''Листинг файлов. По умолчанию — текущая рабочая ветка агента.'''
        br = ctx.resolve(from_my_branch)
        return '[branch: ' + br + ']\n' + gh_list_files(br, subdir)

    def search_code(query: str, subdir: str = '', from_my_branch: bool = True) -> str:
        '''grep по дереву ветки (по умолчанию — рабочей). Возвращает path:line: текст.'''
        br = ctx.resolve(from_my_branch)
        return '[branch: ' + br + ']\n' + gh_search_code(query, br, subdir)

    def read_file(path: str, start_line: int = 1, max_lines: int = 0,
                  from_my_branch: bool = True) -> str:
        '''Чтение файла из рабочей ветки; from_my_branch=False — из base (main).'''
        br = ctx.resolve(from_my_branch)
        return '[branch: ' + br + ']\n' + gh_read_cursor(path, br, start_line, max_lines or None)

    def diff_vs_base() -> str:
        '''Что рабочая ветка меняет относительно base.'''
        d = gh_compare(ctx.base, ctx.work_branch)
        if not d:
            return 'ERROR: сравнение недоступно'
        return ('ahead_by=' + str(d['ahead_by']) + ', behind_by=' + str(d['behind_by']) +
                '\nfiles: ' + ', '.join(d['files']) +
                '\ncommits: ' + ' | '.join(d['commits']))

    return list_files, search_code, read_file, diff_vs_base

## Fix 3. Большие файлы и ноутбуки: курсор вместо слепой обрезки

In [6]:
# Проблема: _clip резал ответ на MAX_TOOL_OUTPUT без указания, где остановился;
# минифицированный ipynb (одна строка на ~49 КБ) нельзя было прочитать вообще.
# Решение: страничное чтение с курсором + инструменты, понимающие структуру ipynb.

def _gh_text(path, branch):
    '''Сырой текст файла ветки, без обрезки.'''
    resp = _gh_request('GET', REPO_PATH + '/contents/' + path, params={'ref': branch})
    if resp.status_code != 200:
        return None
    data = resp.json()
    if data.get('encoding') != 'base64':
        return None
    return base64.b64decode(data['content']).decode('utf-8', errors='replace')


def gh_read_cursor(path, branch, start_line=1, max_lines=None):
    '''Читает срез файла и всегда сообщает, где остановился.

    Хвост ответа: ...[cursor] next_start_line=N из M — агент продолжает чтение
    следующим вызовом, а не теряет остаток файла.
    Отдельно обрабатывается случай одной сверхдлинной строки (минифицированный JSON):
    она отдаётся кусками по символам.
    '''
    text = _gh_text(path, branch)
    if text is None:
        return 'ERROR: ' + path + ' недоступен в ветке ' + branch
    lines = text.splitlines()
    total = len(lines)
    if total == 1 and len(text) > MAX_TOOL_OUTPUT:
        off = max(0, int(start_line) - 1)
        piece = text[off:off + MAX_TOOL_OUTPUT - 200]
        nxt = off + len(piece) + 1
        tail = ('\n...[cursor] one-line file, next_start_line=' + str(nxt) +
                ' (позиция в символах) из ' + str(len(text)))
        return piece + (tail if nxt <= len(text) else '\n...[eof]')
    budget = MAX_TOOL_OUTPUT - 200
    start = max(1, int(start_line))
    limit = total if not max_lines else min(total, start - 1 + int(max_lines))
    out, used, i = [], 0, start - 1
    while i < limit and used + len(lines[i]) + 1 <= budget:
        out.append(lines[i])
        used += len(lines[i]) + 1
        i += 1
    if i == start - 1 and i < limit:
        out.append(lines[i][:budget])
        i += 1
    if i < total:
        tail = '\n...[cursor] next_start_line=' + str(i + 1) + ' из ' + str(total)
    else:
        tail = '\n...[eof] всего строк ' + str(total)
    return '\n'.join(out) + tail


def _cell_src(cell):
    src = cell.get('source')
    return ''.join(src) if isinstance(src, list) else (src or '')


def build_nb_tools(ctx):
    def nb_outline(path: str, from_my_branch: bool = True) -> str:
        '''Оглавление ipynb: индекс, тип, размер и первая строка каждой ячейки.

        Дешёвая карта 50 КБ ноутбука вместо чтения его целиком.
        '''
        br = ctx.resolve(from_my_branch)
        text = _gh_text(path, br)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + br
        try:
            nb = json.loads(text)
        except ValueError as e:
            return 'ERROR: не JSON: ' + str(e)
        rows = []
        for i, c in enumerate(nb.get('cells', [])):
            s = _cell_src(c)
            head = (s.strip().splitlines() or [''])[0][:100]
            rows.append(str(i) + ' [' + c.get('cell_type', '?') + '] ' +
                        str(len(s)) + ' B | ' + head)
        return _clip('cells: ' + str(len(rows)) + '\n' + '\n'.join(rows))

    def nb_cell(path: str, index: int, from_my_branch: bool = True) -> str:
        '''Исходник одной ячейки ipynb по индексу.'''
        br = ctx.resolve(from_my_branch)
        text = _gh_text(path, br)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + br
        cells = json.loads(text).get('cells', [])
        i = int(index)
        if i < 0 or i >= len(cells):
            return 'ERROR: индекс вне диапазона 0..' + str(len(cells) - 1)
        return _clip(_cell_src(cells[i]))

    def nb_replace_cell(path: str, index: int, new_source: str, message: str = '') -> str:
        '''Точечно заменяет исходник одной ячейки: не надо перезаписывать весь JSON.'''
        br = ctx.work_branch
        text = _gh_text(path, br)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + br
        nb = json.loads(text)
        cells = nb.get('cells', [])
        i = int(index)
        if i < 0 or i >= len(cells):
            return 'ERROR: индекс вне диапазона 0..' + str(len(cells) - 1)
        cells[i]['source'] = new_source
        if cells[i].get('cell_type') == 'code':
            cells[i]['outputs'] = []
            cells[i]['execution_count'] = None
        dumped = json.dumps(nb, ensure_ascii=False, indent=1)
        return gh_write_file(path, dumped, br,
                            message or (path + ': заменена ячейка ' + str(i)))

    return nb_outline, nb_cell, nb_replace_cell

## Fix 4. Исполнение кода и создание issues

In [7]:
# Проблема: агент не мог ни запустить свой код (писал вслепую), ни завести issue.
# Решение: изолированный запуск в отдельном процессе с таймаутом + POST /issues.
import subprocess
import sys
import tempfile
import textwrap

RUN_TIMEOUT_DEFAULT = 30


def _run_file(dirpath, filename, argv, timeout_sec):
    try:
        p = subprocess.run(
            [sys.executable, filename] + argv,
            capture_output=True, text=True, timeout=timeout_sec, cwd=dirpath,
            env={'PATH': os.environ.get('PATH', ''), 'HOME': dirpath,
                 'PYTHONDONTWRITEBYTECODE': '1'},
        )
    except subprocess.TimeoutExpired:
        return 'ERROR: превышен таймаут ' + str(timeout_sec) + ' c'
    return _clip('exit=' + str(p.returncode) +
                 '\n--- stdout ---\n' + p.stdout +
                 '\n--- stderr ---\n' + p.stderr)


def run_python(code: str, timeout_sec: int = RUN_TIMEOUT_DEFAULT) -> str:
    '''Выполняет Python-код в отдельном процессе и возвращает exit code, stdout, stderr.

    Окружение урезано (нет GITHUB_TOKEN в env), рабочая директория временная,
    есть таймаут — чтобы агент проверял свои правки, а не гадал.
    '''
    with tempfile.TemporaryDirectory() as d:
        with open(os.path.join(d, 'snippet.py'), 'w', encoding='utf-8') as fh:
            fh.write(textwrap.dedent(code))
        return _run_file(d, 'snippet.py', [], timeout_sec)


def make_run_repo_file(ctx):
    def run_repo_file(path: str, args: str = '', timeout_sec: int = 60,
                      from_my_branch: bool = True) -> str:
        '''Скачивает файл из ветки во временный каталог и запускает его (например, тесты).'''
        br = ctx.resolve(from_my_branch)
        text = _gh_text(path, br)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + br
        with tempfile.TemporaryDirectory() as d:
            name = os.path.basename(path)
            with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
                fh.write(text)
            return _run_file(d, name, args.split() if args else [], timeout_sec)

    return run_repo_file


def create_issue(title: str, body: str = '', labels: str = '') -> str:
    '''Создаёт новый issue в репозитории (нужен scope Issues: Read and write).'''
    payload = {'title': title, 'body': body}
    if labels:
        payload['labels'] = [x.strip() for x in labels.split(',') if x.strip()]
    resp = _gh_request('POST', REPO_PATH + '/issues', json=payload)
    if resp.status_code != 201:
        return _err('create_issue', resp) + '  (нужен scope Issues: RW)'
    return 'Issue создан: ' + resp.json()['html_url']

## Сборка набора инструментов и что поменять в раннере

In [8]:
import functools


def build_toolset(slug, base=None):
    '''Возвращает (ctx, tools) для одной модели. ctx создаётся ОДИН раз и
    переиспользуется во всех раундах — именно это чинит пункт 1 issue #16.'''
    ctx = AgentContext(slug, base)
    create_issue_branch, use_branch, current_branch = make_branch_tools(ctx)
    list_files, search_code, read_file, diff_vs_base = build_read_tools(ctx)
    nb_outline, nb_cell, nb_replace_cell = build_nb_tools(ctx)
    run_repo_file = make_run_repo_file(ctx)

    def write_file(path: str, content: str, commit_message: str = '') -> str:
        '''Полная запись файла в текущую рабочую ветку.'''
        return gh_write_file(path, content, ctx.work_branch, commit_message or None)

    def patch_file(path: str, old_text: str, new_text: str) -> str:
        '''Замена первого вхождения old_text на new_text в файле рабочей ветки.'''
        text = _gh_text(path, ctx.work_branch)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + ctx.work_branch
        if old_text not in text:
            return 'ERROR: фрагмент не найден'
        return gh_write_file(path, text.replace(old_text, new_text, 1), ctx.work_branch,
                             path + ': patch')

    tools = [
        current_branch, create_issue_branch, use_branch,
        list_files, search_code, read_file, diff_vs_base,
        nb_outline, nb_cell, nb_replace_cell,
        write_file, patch_file,
        run_python, run_repo_file, create_issue,
    ]
    return ctx, tools


# Чек-лист изменений в раннере (движок раундов):
# 1. ctx/tools строятся один раз на модель, а не на раунд;
# 2. в системный промпт каждого раунда добавляется ctx.round_header(i, n)
#    -> модель видит текущую ветку и не пишет в дефолтную;
# 3. состояние дублируется в kbench_agent_state.json (переживает рестарт ядра);
# 4. функции-инструменты регистрируются с functools.wraps, иначе схема
#    вырождается в {args, kwargs} (известный баг проекта);
# 5. критерий github_issue_resolution дополняется проверкой,
#    что коммиты попали именно в <slug>-issue-<N>, а не в <slug>.

## 3. Бюджет шагов (как в v1, `functools.wraps` обязателен)

In [9]:
class StepBudget:
    def __init__(self, limit, log=None):
        self.limit, self.used = limit, 0
        # log — общий список на ВСЮ задачу (все раунды), а не только на этот
        # раунд: сюда пишет with_budget при каждом реальном вызове инструмента.
        # Это и есть память между раундами — объективная, не зависящая от
        # того, напишет ли модель честный текстовый самоотчёт.
        self.log = log if log is not None else []

    def consume(self):
        if self.used >= self.limit:
            return False
        self.used += 1
        return True

    @property
    def remaining(self):
        return max(0, self.limit - self.used)


def with_budget(tool_fn, budget):
    """functools.wraps обязателен: иначе kbench не построит схему параметров."""
    @functools.wraps(tool_fn)
    def wrapped(*args, **kwargs):
        if not budget.consume():
            return (
                f"ERROR: бюджет шагов на раунд исчерпан ({budget.limit}). "
                "Заверши раунд текстовым ответом."
            )
        try:
            result = tool_fn(*args, **kwargs)
        except Exception as e:  # noqa: BLE001 — агент должен видеть текст, а не падение
            result = f"ERROR: {type(e).__name__}: {e}"
        # фиксируем реальный вызов в общий лог задачи (переживает смену раунда)
        budget.log.append({
            "tool": tool_fn.__name__,
            "args": args,
            "kwargs": kwargs,
            "result_preview": str(result)[:20000],
        })
        return result
    return wrapped


# kaggle_benchmarks.tools.native.native_tool_agent сам ограничивает один вызов
# llm.prompt(...) 10 "раундами" инструментов и кидает ToolInvocationLimitExhausted
# при превышении — держим бюджет заметно ниже этого внутреннего лимита.
STEPS_PER_ROUND = 10
MAX_ROUNDS = 4


## 3b. GitHub Codespaces: инфраструктура (адаптировано из `codespaces_utils.py`)

Даёт агенту три новые возможности поверх обычного git-flow из раздела 4:

1. **`start_codespace()`** — поднимает codespace на текущей рабочей ветке агента
   (`state["branch"]`, тот же guard, что у `write_file`/`patch_file`) и
   возвращает `web_url` — ссылку, которую можно прислать человеку, чтобы тот
   открыл рабочий код в браузере (VS Code for the Web) и проверил его сам.
2. **`run_in_codespace(command, timeout_sec)`** — выполняет команду (тесты,
   запуск проекта) внутри уже поднятого codespace через `gh codespace ssh` и
   возвращает exit code/stdout/stderr — самостоятельная проверка агентом до
   отчёта в issue.
3. **`stop_codespace()`** — останавливает codespace после того, как человек
   закончил проверку (или агент закончил тестирование). Не удаляет его —
   можно поднять заново тем же `start_codespace()`.

Требования и ограничения:
- Токену нужен отдельный scope **Codespaces: Read and write** — без него
  `start_codespace`/`stop_codespace` упадут 403/404 (та же история, что была
  с `Contents: Read and write` для `create_branch`/`write_file`).
- `run_in_codespace` требует установленного и авторизованного `gh` CLI
  (ставится ячейкой ниже); REST API GitHub не умеет выполнять команды внутри
  codespace — только жизненный цикл (создать/статус/остановить).
- Guard на одновременность: не больше одного активного codespace на агента
  (имя хранится в `state["codespace"]`) — повторный `start_codespace()`
  просто возвращает уже существующий вместо траты квоты на дубликаты.
- GitHub сам останавливает простаивающий codespace по `idle_timeout`
  (ниже выставлен в 30 минут) — подстраховка на случай, если агент забудет
  вызвать `stop_codespace()`.


In [10]:
# ---------------------------------------------------------------------------
# Codespaces: установка и авторизация gh CLI (адаптировано из codespaces_utils.py)
# ---------------------------------------------------------------------------
import shutil
import subprocess


def _ensure_gh_cli():
    """Ставит gh CLI, если его нет, и авторизует тем же GITHUB_TOKEN."""
    if shutil.which("gh") is None:
        steps = [
            "apt-get update -qq",
            "apt-get install -y -qq curl",
            "curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg "
            "-o /usr/share/keyrings/githubcli-archive-keyring.gpg",
            "chmod go+r /usr/share/keyrings/githubcli-archive-keyring.gpg",
            'bash -c \'echo "deb [arch=$(dpkg --print-architecture) '
            'signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] '
            'https://cli.github.com/packages stable main" '
            "> /etc/apt/sources.list.d/github-cli.list'",
            "apt-get update -qq",
            "apt-get install -y -qq gh",
        ]
        for cmd in steps:
            r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
            if r.returncode != 0:
                print(f"gh CLI install: шаг провалился ({cmd[:40]}...): {r.stderr[:300]}")
                return False
    # gh CLI сам подхватывает GH_TOKEN из окружения — отдельный `gh auth login` не нужен
    os.environ["GH_TOKEN"] = GITHUB_TOKEN
    check = subprocess.run(["gh", "auth", "status"], capture_output=True, text=True)
    ok = check.returncode == 0
    print("gh CLI готов и авторизован." if ok else (check.stdout + check.stderr)[:500])
    return ok


GH_CLI_READY = _ensure_gh_cli()


gh CLI готов и авторизован.


In [11]:
# ---------------------------------------------------------------------------
# Codespaces: жизненный цикл через REST (create/status/stop, переиспользует
# _gh_request) + выполнение команд через gh CLI (REST этого не умеет)
# ---------------------------------------------------------------------------
import time as _time

CODESPACE_MACHINE = "basicLinux32gb"   # самая дешёвая машина по умолчанию
CODESPACE_IDLE_TIMEOUT_MIN = 30        # подстраховка: GitHub сам остановит простой codespace


def gh_create_codespace(branch):
    """Создаёт codespace на указанной ветке репозитория (REST API)."""
    resp = _gh_request(
        "POST", f"{REPO_PATH}/codespaces",
        json={"ref": branch, "machine": CODESPACE_MACHINE,
              "idle_timeout_minutes": CODESPACE_IDLE_TIMEOUT_MIN},
    )
    if resp.status_code not in (200, 201):
        return None, _err("create_codespace", resp) + "  (нужен scope Codespaces: RW)"
    return resp.json(), None


def gh_get_codespace(name):
    resp = _gh_request("GET", f"/user/codespaces/{name}")
    return resp.json() if resp.status_code == 200 else None


def gh_wait_codespace_available(name, timeout_sec=180):
    """Опрашивает статус codespace, пока он не станет Available или не истечёт таймаут."""
    deadline = _time.time() + timeout_sec
    while _time.time() < deadline:
        data = gh_get_codespace(name)
        state = (data or {}).get("state")
        if state == "Available":
            return True, state
        if state in ("Failed", "Deleted", "Shutdown", "Unavailable"):
            return False, state
        _time.sleep(5)
    return False, "Timeout"


def gh_stop_codespace(name):
    resp = _gh_request("POST", f"/user/codespaces/{name}/stop")
    if resp.status_code not in (200, 202):
        return _err("stop_codespace", resp)
    return f"Codespace {name} остановлен."


# state=Available в REST API значит только то, что поднялась VM/контейнер
# codespace. sshd внутри него стартует отдельно (после onCreate/postCreate)
# и может быть не готов ещё несколько секунд/десятков секунд после Available,
# особенно на первом запуске. gh в этом случае возвращает
# "error getting ssh server details: failed to start SSH server" — это
# транзиентная ошибка, отличная от "sshd вообще нет в образе" только по
# факту повторяемости, поэтому ретраим с бэкоффом, а не падаем с первой попытки.
SSH_NOT_READY_MARKERS = (
    "error getting ssh server details",
    "failed to start ssh server",
)
SSH_READY_RETRIES = 6
SSH_READY_RETRY_DELAY_SEC = 10


def gh_exec_in_codespace(name, command, timeout_sec=120):
    """Выполняет команду внутри codespace через `gh codespace ssh` (нужен gh CLI).

    Ретраит отдельно на случай, когда VM уже Available, а sshd внутри
    контейнера ещё не поднялся (см. SSH_NOT_READY_MARKERS выше).
    """
    if not GH_CLI_READY:
        return "ERROR: gh CLI недоступен/не авторизован — run_in_codespace не работает"

    last_output = ""
    for attempt in range(1, SSH_READY_RETRIES + 1):
        try:
            # gh codespace ssh НЕ принимает флаг --command: команда передаётся
            # только позиционно после `--`. Заворачиваем в `bash -lc "..."`,
            # иначе gh/ssh под капотом раздробит command на отдельные
            # позиционные аргументы по пробелам и сломает пайпы/кавычки/&&.
            r = subprocess.run(
                ["gh", "codespace", "ssh", "--codespace", name,
                 "--", "bash", "-lc", command],
                capture_output=True, text=True, timeout=timeout_sec,
            )
            last_output = _clip(
                f"exit={r.returncode}\n--- stdout ---\n{r.stdout}\n--- stderr ---\n{r.stderr}"
            )
            ssh_not_ready = r.returncode != 0 and any(
                marker in r.stderr.lower() for marker in SSH_NOT_READY_MARKERS
            )
            if not ssh_not_ready:
                return last_output
            if attempt < SSH_READY_RETRIES:
                _time.sleep(SSH_READY_RETRY_DELAY_SEC)
        except subprocess.TimeoutExpired:
            return f"ERROR: команда не уложилась в {timeout_sec} c"

    return (
        f"ERROR: sshd в codespace так и не поднялся за "
        f"{SSH_READY_RETRIES * SSH_READY_RETRY_DELAY_SEC} c после Available.\n"
        f"Похоже, sshd отсутствует в образе — проверьте .devcontainer/devcontainer.json "
        f"(нужна фича ghcr.io/devcontainers/features/sshd:1) или Dockerfile репозитория.\n"
        f"Последний вывод:\n{last_output}"
    )


def gh_list_my_codespaces():
    """Дешёвая проверка scope Codespaces без создания codespace (для smoke_test)."""
    resp = _gh_request("GET", "/user/codespaces")
    if resp.status_code != 200:
        return f"ERROR: list_codespaces failed ({resp.status_code}): {resp.text[:200]}"
    return f"{resp.json().get('total_count', 0)} codespace(s) на аккаунте"


In [ ]:
# ---------------------------------------------------------------------------
# Smoke-тест gh_exec_in_codespace: поднимаем codespace на базовой ветке,
# гоняем несколько команд (включая случай с пробелами/кавычками/&&, из-за
# которого была правка выше) и печатаем итог — прошёл тест или упал.
# Ничего не пишет в репозиторий и не трогает ветки моделей.
# ---------------------------------------------------------------------------

CODESPACE_SMOKE_TEST_COMMANDS = [
    ("node -v", "node -v"),
    ("простая команда с пробелами в аргументах", 'echo "hello   world"'),
    ("команда с && (несколько шагов в одном вызове)", "echo one && echo two"),
    ("команда с одинарными и двойными кавычками вперемешку",
     'echo "it\'s a test" && echo \'second "part"\''),
]


def run_codespace_smoke_test():
    if not GH_CLI_READY:
        print("SMOKE TEST: ПРОПУЩЕН — gh CLI недоступен/не авторизован")
        return False

    print(f"Поднимаю codespace на ветке {GITHUB_BASE_BRANCH} для smoke-теста...")
    data, err = gh_create_codespace(GITHUB_BASE_BRANCH)
    if err:
        print(f"SMOKE TEST: ПРОВАЛЕН — не удалось создать codespace: {err}")
        return False

    name = data["name"]
    print(f"Codespace создан: {name}. Жду готовности...")
    ok, cs_state = gh_wait_codespace_available(name)
    if not ok:
        print(f"SMOKE TEST: ПРОВАЛЕН — codespace не стал Available (state={cs_state})")
        gh_stop_codespace(name)
        return False

    print(f"Codespace готов (state={cs_state}). Прогоняю тестовые команды...\n")

    all_passed = True
    for label, cmd in CODESPACE_SMOKE_TEST_COMMANDS:
        result = gh_exec_in_codespace(name, cmd, timeout_sec=60)
        exit_line = result.splitlines()[0] if result else ""
        passed = exit_line.strip() == "exit=0" and "unknown flag" not in result
        status = "OK" if passed else "FAIL"
        all_passed = all_passed and passed
        print(f"[{status}] {label}\n  команда: {cmd}\n  {exit_line}")
        if not passed:
            print(f"  полный вывод:\n{result}")
            if "sshd в codespace так и не поднялся" in result:
                print("  -> похоже, в devcontainer.json репозитория нет фичи sshd, "
                      "см. подсказку в тексте ошибки выше")
        print()

    print("Останавливаю codespace...")
    print(gh_stop_codespace(name))

    verdict = "SMOKE TEST: ПРОЙДЕН — run_in_codespace работает" if all_passed \
        else "SMOKE TEST: ПРОВАЛЕН — см. FAIL выше"
    print(f"\n{verdict}")
    return all_passed


CODESPACE_SMOKE_TEST_PASSED = run_codespace_smoke_test()


## 4. Расширенный набор инструментов агента

Guard: агент может писать **только** в ветки с префиксом собственного slug.
Имя ветки нигде не приходит от модели напрямую — оно вычисляется из номера issue.

In [ ]:
def slugify_model_id(model_id):
    return re.sub(r"[^a-z0-9]+", "-", model_id.lower()).strip("-")


MAX_ISSUES_CREATED_PER_RUN = 3  # не давать агенту уйти в репортинг вместо решения задачи


def make_github_tools(slug, budget, allow_issue_write=True):
    """Возвращает список инструментов, замкнутых на slug модели."""
    state = {"branch": slug, "codespace": None, "issues_created": 0}
    # ^ рабочая ветка + активный codespace + счётчик заведённых агентом issues

    def _guard(branch):
        if branch != slug and not branch.startswith(slug + "-issue-"):
            raise PermissionError(f"запись в ветку {branch} запрещена (не твой префикс)")
        return branch

    def list_branches() -> str:
        """List all branch names in the repository."""
        return gh_list_branches()

    def create_branch() -> str:
        """Create your main assigned branch (idempotent)."""
        state["branch"] = slug
        return gh_create_branch(slug)

    def create_issue_branch(issue_number: int) -> str:
        """Create and select a dedicated branch for an issue: <your-slug>-issue-<N>."""
        name = _guard(f"{slug}-issue-{int(issue_number)}")
        result = gh_create_branch(name)
        if result.startswith("ERROR:"):
            return result  # рабочую ветку не переключаем — не маскируем неудачное создание
        state["branch"] = name
        return result + f" Текущая рабочая ветка: {name}"

    def list_files(subdir: str = "") -> str:
        """List all files in the base branch, optionally filtered by path prefix."""
        return gh_list_files(GITHUB_BASE_BRANCH, subdir)

    def read_file(path: str, from_my_branch: bool = False,
                  start_line: int = 1, max_lines: int = 0) -> str:
        """Read a file. from_my_branch=True reads your working branch instead of base.
        Use start_line/max_lines to read only a slice of a large file."""
        branch = state["branch"] if from_my_branch else GITHUB_BASE_BRANCH
        return gh_read_file(path, branch, start_line, max_lines or None)

    def search_code(query: str, subdir: str = "") -> str:
        """Grep the base branch for a substring. Returns 'path:line: text' matches."""
        return gh_search_code(query, GITHUB_BASE_BRANCH, subdir)

    def write_file(path: str, content: str, commit_message: str = "") -> str:
        """Create or overwrite a file with FULL content on your working branch."""
        return gh_write_file(path, content, _guard(state["branch"]), commit_message or None)

    def patch_file(path: str, old_text: str, new_text: str) -> str:
        """Replace the first occurrence of old_text with new_text in a file
        on your working branch. Cheaper and safer than rewriting a whole file."""
        branch = _guard(state["branch"])
        current = gh_read_file(path, branch)
        if current.startswith("ERROR:"):
            return current
        if old_text not in current:
            return "ERROR: old_text не найден — прочитай файл и повтори точную подстроку"
        updated = current.replace(old_text, new_text, 1)
        return gh_write_file(path, updated, branch, f"{path}: patch")

    def delete_file(path: str) -> str:
        """Delete a file on your working branch."""
        return gh_delete_file(path, _guard(state["branch"]))

    def diff_vs_base() -> str:
        """Show how your working branch differs from the base branch."""
        cmp = gh_compare(GITHUB_BASE_BRANCH, state["branch"])
        if cmp is None:
            return f"ERROR: не удалось сравнить {state['branch']} с {GITHUB_BASE_BRANCH}"
        return (
            f"branch={state['branch']} ahead_by={cmp['ahead_by']} behind_by={cmp['behind_by']}\n"
            f"files: {', '.join(cmp['files']) or '-'}\n"
            f"commits: {'; '.join(cmp['commits']) or '-'}"
        )

    def start_codespace() -> str:
        """Start (or reuse) a Codespace on your current working branch.
        Returns the codespace name and a web_url you can share with the
        human so they can open the working code in the browser and test
        it themselves."""
        branch = state["branch"]
        existing = state.get("codespace")
        if existing:
            data = gh_get_codespace(existing)
            if data and data.get("state") not in ("Shutdown", "Deleted", "Failed"):
                return (f"Codespace {existing} уже активен (state={data.get('state')}). "
                        f"Ссылка: {data.get('web_url')}")
        data, err = gh_create_codespace(branch)
        if err:
            return err
        name = data["name"]
        state["codespace"] = name
        ok, cs_state = gh_wait_codespace_available(name)
        status = "готов" if ok else f"не готов (state={cs_state})"
        return (f"Codespace {name} создан на ветке {branch}, {status}.\n"
                f"Ссылка для проверки человеком: {data.get('web_url')}")

    def run_in_codespace(command: str, timeout_sec: int = 120) -> str:
        """Run a shell command inside your active Codespace (tests, build,
        run the project) and return exit code / stdout / stderr.
        Call start_codespace() first."""
        name = state.get("codespace")
        if not name:
            return "ERROR: сначала вызови start_codespace()"
        return gh_exec_in_codespace(name, command, timeout_sec)

    def stop_codespace() -> str:
        """Stop your active Codespace — call this once the human confirms
        they finished testing it, or once you're done verifying your
        changes yourself. Safe to call even if nothing is running."""
        name = state.get("codespace")
        if not name:
            return "Codespace не запущен."
        result = gh_stop_codespace(name)
        state["codespace"] = None
        return result


    def list_issues(state_filter: str = "all", labels: str = "") -> str:
        """List ALL issues (open and closed) with number, state, title and labels."""
        return gh_list_issues(state_filter, labels or None)

    def get_issue(issue_number: int) -> str:
        """Read the full body and all comments of one issue."""
        return gh_get_issue(int(issue_number))

    def comment_issue(issue_number: int, body: str) -> str:
        """Post a comment on an issue."""
        return gh_comment_issue(int(issue_number), body)

    def report_work(issue_number: int, summary: str) -> str:
        """Post a structured report on the issue: link to your branch, diff link,
        changed files and your summary. Use this to close the loop on a task."""
        branch = state["branch"]
        cmp = gh_compare(GITHUB_BASE_BRANCH, branch) or {"files": [], "commits": [], "ahead_by": 0}
        body = (
            f"### Работа по issue #{int(issue_number)}\n\n"
            f"{summary}\n\n"
            f"**Ветка:** [`{branch}`]({REPO_URL}/tree/{branch})\n"
            f"**Diff:** {REPO_URL}/compare/{GITHUB_BASE_BRANCH}...{branch}\n"
            f"**Коммитов впереди `{GITHUB_BASE_BRANCH}`:** {cmp['ahead_by']}\n"
            f"**Изменённые файлы:** "
            + (", ".join(f"`{f}`" for f in cmp["files"]) or "—")
            + "\n\n<sub>Автоматический отчёт агента.</sub>"
        )
        return gh_comment_issue(int(issue_number), body)

    def create_issue(title: str, body: str = "", priority: str = "p2") -> str:
        """File a NEW issue for a problem you found OUTSIDE the scope of your
        current assigned issue — never as a substitute for solving it.
        Set priority honestly: 'p0' blocks using the repo/benchmark,
        'p1' is a real problem with a workaround, 'p2' is minor/nice-to-have.
        Limited to a few per run so you stay focused on your actual task."""
        if state["issues_created"] >= MAX_ISSUES_CREATED_PER_RUN:
            return (f"ERROR: лимит {MAX_ISSUES_CREATED_PER_RUN} новых issues на "
                     "один прогон исчерпан — сосредоточься на своей задаче")
        priority = (priority or "p2").lower().strip()
        if priority not in ("p0", "p1", "p2"):
            priority = "p2"
        result = gh_create_issue(title, body, [f"priority:{priority}"])
        if not result.startswith("ERROR:"):
            state["issues_created"] += 1
        return result

    def open_pull_request(title: str, body: str = "") -> str:
        """Open a pull request from your working branch into the base branch."""
        return gh_create_pull_request(state["branch"], GITHUB_BASE_BRANCH, title, body)

    def report_no_action_needed(issue_number: int, reason: str) -> str:
        """Use INSTEAD of report_work when, after actually reading the issue
        (get_issue) and the relevant code, you conclude the repository code
        genuinely does NOT need any change — e.g. the issue explicitly tells
        agents/models not to touch the repo, or describes a problem outside
        the repo's own code (tooling/environment bug, duplicate, etc).
        Explain WHY clearly and specifically: a separate judge model will
        compare your reasoning against the issue text, so a lazy, generic or
        false justification will be rejected and the task will still count
        as failed. Never use this just to avoid doing real work."""
        body = (
            f"### Без изменений кода по issue #{int(issue_number)}\n\n"
            f"{reason}\n\n"
            f"{NO_ACTION_MARKER}\n"
            "<sub>Автоматический отчёт агента (путь \"без изменений\").</sub>"
        )
        return gh_comment_issue(int(issue_number), body)

    tools = [
        list_branches, create_branch, create_issue_branch,
        list_files, read_file, search_code,
        write_file, patch_file, delete_file, diff_vs_base,
        start_codespace, run_in_codespace, stop_codespace,
    ]
    if allow_issue_write:
        tools += [list_issues, get_issue, comment_issue, report_work,
                  report_no_action_needed, create_issue, open_pull_request]
    else:
        tools += [list_issues, get_issue]
    return [with_budget(t, budget) for t in tools]

## 4c. Приоритет issues и создание новых

**Почему это вообще нужно.** У каждой пары модель×issue свой отдельный бюджет шагов/раундов (см. `MAX_ROUNDS`/`STEPS_PER_ROUND`) — так что в рамках ОДНОЙ задачи агент не выбирает, какой issue решать: номер ему даёт раннер (раздел 6). Но раннер прогоняет **все** открытые issues подряд для каждой модели, а прогон может оборваться раньше времени (квота Kaggle Model Proxy, время сессии, rate limit) — то есть не факт, что модель дойдёт до issue в конце списка. Поэтому важен **порядок**, в котором issues идут в очередь, а не попытка впихнуть все issues в бюджет одной задачи.

**Схема приоритета** — лейбл на issue:
- `priority:p0` — блокирует использование репозитория/бенчмарка, решать в первую очередь;
- `priority:p1` — реальная проблема, но есть обходной путь;
- `priority:p2` — мелочь/улучшение, можно отложить;
- без лейбла — issue не протриажен; `gh_list_issues`/раннер ставят ему средний ранг (между p1 и p2), чтобы такие issues не проваливались в конец очереди, но и не перебивали явный `p0`.

**Где это применяется:**
1. `list_issues()` (инструмент агента) сортирует вывод по приоритету и печатает тег `[P0]`/`[P1]`/`[P2]`/`[??]` перед каждым issue.
2. Раннер в разделе 6 сортирует `open_issues` по приоритету **перед** циклом по моделям — так P0-issues проверяются на всех моделях раньше P2, даже если сам прогон оборвётся на середине списка.
3. Новый инструмент `create_issue(title, body, priority)` — агент заводит issue под проблему, которую нашёл **вне рамок своей текущей задачи**, и обязан честно проставить приоритет по критериям выше. Это не значит, что агент сам берётся её чинить в этом же прогоне — новый issue просто встанет в очередь для будущего запуска раннера (см. раздел 6: список `open_issues` фиксируется **до** начала цикла, так что issue, созданный во время прогона, в этом же прогоне обработан не будет).
4. Лимит `MAX_ISSUES_CREATED_PER_RUN` — не больше нескольких новых issues за один прогон одной задачи, чтобы агент не тратил бюджет шагов на репортинг вместо решения назначенной ему issue.


## 5. Задача `github_issue_resolution`

Успех засчитывается **только** если одновременно:
1. ветка `<slug>-issue-<N>` существует и опережает `main` хотя бы на 1 коммит;
2. в issue есть комментарий со ссылкой на эту ветку.

In [ ]:
import kaggle_benchmarks as kbench  # noqa: E402
import pandas as pd  # noqa: E402

ISSUE_SYSTEM_PROMPT = (
    "Ты — инженер-агент, работающий с реальным GitHub-репозиторием только через "
    "инструменты. Твой цикл работы: прочитать issue -> изучить релевантные файлы "
    "(list_files/search_code/read_file) -> создать ветку под issue "
    "(create_issue_branch) -> внести изменения (write_file/patch_file) -> "
    "проверить (diff_vs_base) -> отчитаться в issue (report_work). "
    "Никогда не выдумывай содержимое файлов — сначала читай.\n\n"
    "Если после ЧЕСТНОГО изучения issue и кода ты убедился, что правка кода "
    "репозитория НЕ требуется (например, issue прямо просит агентов её не "
    "трогать, или описывает проблему вне кода репозитория) — используй "
    "report_no_action_needed(issue_number, reason) ВМЕСТО report_work и "
    "распиши причину конкретно. Твоё обоснование проверит отдельная модель-"
    "судья, сверяя его с текстом issue — общими фразами не отделаться, и "
    "использовать это как повод ничего не делать, когда правка реально нужна, "
    "нельзя.\n\n"
    "Дополнительно у тебя есть реальная облачная среда (GitHub Codespaces) поверх "
    "твоей рабочей ветки — используй её не вместо, а в дополнение к diff_vs_base:\n"
    "- start_codespace() — поднять codespace на твоей ветке и получить ссылку, "
    "которую можно прислать человеку, чтобы он сам открыл и проверил рабочий код;\n"
    "- run_in_codespace(command) — самому запустить проект/тесты в этой среде и "
    "посмотреть на реальный результат перед отчётом;\n"
    "- stop_codespace() — обязательно останови codespace, когда человек подтвердит, "
    "что закончил проверку, или когда сам закончил тестирование — не оставляй его "
    "висеть без необходимости.\n\n"
    "Про другие issues репозитория: номер твоей задачи уже выбран за тебя, "
    "не отвлекайся на остальные. list_issues() показывает их с приоритетом "
    "([P0]/[P1]/[P2]/[??], сортировка от важного к неважному). Если в процессе "
    "работы ты находишь ОТДЕЛЬНУЮ проблему вне рамок своей задачи — заведи под "
    "неё create_issue(title, body, priority), честно оценив приоритет по "
    "критериям: p0 — блокирует использование репозитория, p1 — реальная "
    "проблема с обходным путём, p2 — мелочь. Это не замена решению своей "
    "задачи, а всего лишь пометка на будущее; лимит — несколько issues за прогон."
)

ISSUE_ROUNDS_LOG = []

# Порог сырого лога вызовов (в символах), после которого он схлопывается в
# резюме через отдельный вызов LLM, а не растёт неограниченно из раунда в раунд.
CALL_LOG_CHAR_LIMIT = 120000

SUMMARIZE_SYSTEM_PROMPT = (
    "Ты сжимаешь журнал вызовов инструментов GitHub-агента в компактный статус. "
    "На входе — предыдущее краткое резюме (может быть пустым) и новые сырые записи "
    "вызовов инструментов. Выдай ОДИН компактный абзац на русском: что уже сделано "
    "(создана ли ветка, какие файлы записаны, был ли report_work), без пересказа "
    "промежуточных read/list-вызовов, если они не дали важной информации. "
    "Не выдумывай факты, которых нет в записях."
)

# --- Судья для пути "изменения не нужны" (report_no_action_needed) ---------
# Независимая проверка: сам агент не может засчитать себе "изменения не
# нужны" просто заявив это — обоснование сверяется с текстом issue отдельной
# моделью. Судья намеренно НЕ та же модель, что тестируется на этом issue,
# иначе проверка вырождается в "модель подтверждает сама себя".

JUDGE_SYSTEM_PROMPT = (
    "Ты — независимый судья, проверяющий решение GitHub-агента не вносить "
    "изменения в код репозитория. Тебе даны полный текст issue (с комментариями) "
    "и объяснение агента, почему правка не нужна. Ответь ОДНИМ словом:\n"
    "ДА — если объяснение агента разумно обосновано текстом issue (issue явно "
    "не требует правки кода репозитория, прямо просит его не трогать, либо "
    "описывает проблему вне кода репозитория, а объяснение агента это "
    "корректно отражает);\n"
    "НЕТ — если агент, похоже, уклонился от реальной работы: issue описывает "
    "конкретную решаемую проблему в коде репозитория, а объяснение агента "
    "общее, надуманное или не соответствует содержанию issue.\n"
    "Не объясняй ответ и не добавляй ничего кроме ДА или НЕТ."
)


def pick_judge_model_id(tested_model_id):
    """Судья не должен быть той же моделью, что тестируется на этом issue —
    иначе она просто подтверждает сама себя. Берём первую другую модель из
    MODELS (глобальный список, определяется в разделе с прогоном)."""
    for mid in MODELS:
        if mid != tested_model_id:
            return mid
    return tested_model_id  # только одна модель в MODELS — самопроверка неизбежна


def judge_no_action(tested_model_id, issue_text, reason):
    judge_model_id = pick_judge_model_id(tested_model_id)
    judge_llm = kbench.llms[judge_model_id]
    prompt = (
        f"{JUDGE_SYSTEM_PROMPT}\n\n--- ISSUE (с комментариями) ---\n{issue_text}\n\n"
        f"--- ОБЪЯСНЕНИЕ АГЕНТА ({tested_model_id}) ---\n{reason}"
    )
    try:
        _jitter()
        verdict = judge_llm.prompt(prompt).strip().upper()
        print(f"[JUDGE:{judge_model_id}] вердикт по {tested_model_id}: {verdict[:20]!r}")
        return verdict.startswith("ДА") or verdict.startswith("YES")
    except Exception as e:  # noqa: BLE001 — сбой судьи не должен падать всей задачей
        print(f"[JUDGE] ошибка при проверке no-action: {type(e).__name__}: {e}")
        return False  # при сбое судьи — не даём бесплатный зачёт


def format_log_entries(entries):
    return "\n".join(
        f"- {e['tool']}({e['kwargs']}) -> {e['result_preview']}" for e in entries
    ) or "(вызовов не было)"


def summarize_call_log(llm, log_summary, entries):
    """Сжимает call_log через LLM. При сбое — безопасный фолбэк вместо падения задачи."""
    raw_text = format_log_entries(entries)
    prompt = (
        f"{SUMMARIZE_SYSTEM_PROMPT}\n\n"
        f"Предыдущее резюме:\n{log_summary or '(пусто, это первое сжатие)'}\n\n"
        f"Новые записи с прошлого сжатия:\n{raw_text}\n\n"
        "Дай новое единое резюме, учитывающее и предыдущее, и новые записи."
    )
    try:
        _jitter()
        # без tools — это чисто текстовый вызов на суммаризацию, не агентный шаг
        summary = llm.prompt(prompt)
        return summary.strip()
    except Exception as e:  # noqa: BLE001 — сжатие не должно ронять задачу
        print(f"[SUMMARIZE] ошибка: {type(e).__name__}: {e}")
        # фолбэк: наивно схлопываем в список имён вызванных инструментов
        fallback = (
            f"{log_summary or ''}\n[сжатие через LLM не удалось "
            f"({type(e).__name__}): оставлены только имена вызванных инструментов] "
            + ", ".join(e["tool"] for e in entries)
        ).strip()
        return fallback


def _prompt_with_retry(llm, msg, tools, round_num):
    """Один вызов llm.prompt с джиттером и одним ретраем на похожие на
    временные ошибки (rate limit / quota / 429). Полный текст исключения
    печатается сразу — раньше он был виден только обрезанным внутри
    self_report, что мешало понять настоящую причину сбоя."""
    _jitter()
    try:
        return llm.prompt(msg, tools=tools), None
    except Exception as e:  # noqa: BLE001
        print(f"[EXC] раунд {round_num}: {type(e).__name__}: {e}")
        if not _is_retryable(e):
            return None, e
        print(f"[EXC] раунд {round_num}: похоже на временную ошибку, ретраю через паузу")
        _jitter()
        try:
            return llm.prompt(msg, tools=tools), None
        except Exception as e2:  # noqa: BLE001
            print(f"[EXC] раунд {round_num} (после ретрая): {type(e2).__name__}: {e2}")
            return None, e2


def run_rounds(llm, make_round_tools, task_check_fn, task_context):
    self_report = None
    call_log = []      # свежие сырые записи с прошлого сжатия (общие на все раунды)
    log_summary = ""   # накопленное сжатое резюме более ранних записей

    for round_num in range(1, MAX_ROUNDS + 1):
        budget = StepBudget(STEPS_PER_ROUND, log=call_log)
        tools = make_round_tools(budget)
        if round_num == 1:
            msg = (
                f"{task_context}\n\nУ тебя до {STEPS_PER_ROUND} вызовов инструментов "
                f"на раунд и до {MAX_ROUNDS} раундов."
            )
        else:
            raw_text = format_log_entries(call_log)
            # проверяем размер ИМЕННО перед вставкой в промпт следующего раунда —
            # сжимаем только когда лог реально разросся, а не по расписанию
            if len(raw_text) > CALL_LOG_CHAR_LIMIT:
                log_summary = summarize_call_log(llm, log_summary, call_log)
                call_log.clear()  # схлопнули в log_summary, копим заново
                raw_text = format_log_entries(call_log)

            history = (
                (f"Резюме прошлых раундов:\n{log_summary}\n\n" if log_summary else "")
                + f"Вызовы с прошлого резюме:\n{raw_text}"
            )
            msg = (
                f"Раунд {round_num}/{MAX_ROUNDS}. Новый бюджет: {STEPS_PER_ROUND} "
                f"вызовов.\n{history}\n"
                "Не повторяй уже сделанные вызовы без необходимости, продолжай "
                "со следующего логического шага."
            )
        if round_num == MAX_ROUNDS:
            msg += (
                "\n\nПОСЛЕДНИЙ РАУНД. В конце добавь блок:\n===SELF_REPORT===\n"
                "готово_процентов: <0-100>\nосталось_сделать: <кратко>\n"
                "нужно_ещё_шагов: <число>\n===END_SELF_REPORT==="
            )

        response, exc = _prompt_with_retry(llm, msg, tools, round_num)
        if exc is not None:
            # раунд сорвался (например ToolInvocationLimitExhausted, quota,
            # PermissionDenied) — не роняем всю задачу, переходим к следующему
            # раунду; полный текст ошибки уже напечатан в _prompt_with_retry
            self_report = f"раунд {round_num} прерван исключением: {type(exc).__name__}: {exc}"
            if task_check_fn():
                return True, round_num, self_report
            continue

        m = re.search(r"===SELF_REPORT===(.*?)===END_SELF_REPORT===", response, re.DOTALL)
        if m:
            self_report = m.group(1).strip()
        if task_check_fn():
            return True, round_num, self_report
    return False, MAX_ROUNDS, self_report


def check_issue_solved(model_id, slug, issue_number, issue_text):
    """Два равноправных пути зачёта:
    1) обычная правка кода — коммит(ы) в ветке + комментарий со ссылкой на неё;
    2) report_no_action_needed с обоснованием, подтверждённым независимым
       судьёй (см. judge_no_action) — для issue вроде "не требует правки кода".
    Раньше был только путь (1), из-за чего честное "код менять не нужно"
    гарантированно проваливало проверку (см. issue #9 в прошлых прогонах)."""
    branch = f"{slug}-issue-{issue_number}"
    try:
        comments = _gh_paginate(f"{REPO_PATH}/issues/{issue_number}/comments")
    except RuntimeError:
        return False

    cmp = gh_compare(GITHUB_BASE_BRANCH, branch)
    if cmp and cmp["ahead_by"] >= 1 and any(branch in (c.get("body") or "") for c in comments):
        return True

    no_action_comments = [c for c in comments if NO_ACTION_MARKER in (c.get("body") or "")]
    if no_action_comments:
        reason = no_action_comments[-1].get("body") or ""
        return judge_no_action(model_id, issue_text, reason)

    return False


@kbench.task(name="github_issue_resolution_new")
def github_issue_resolution_eval(llm, model_id: str, issue_number: int) -> bool:
    slug = slugify_model_id(model_id)
    issue_number = int(issue_number)
    branch = f"{slug}-issue-{issue_number}"

    # Своя копия текста issue (не зависит от того, что и как агент прочитал
    # через get_issue) — нужна как вход для судьи в пути "изменения не нужны".
    issue_text = gh_get_issue(issue_number)

    task_context = (
        f"Репозиторий: {GITHUB_OWNER}/{GITHUB_REPO} (база: {GITHUB_BASE_BRANCH}).\n"
        f"Задача: разбери issue #{issue_number}.\n"
        "1. get_issue — прочитай задачу целиком, включая комментарии.\n"
        "2. list_files / search_code / read_file — изучи затронутый код.\n"
        f"3. create_issue_branch({issue_number}) — создай ветку {branch}.\n"
        "4. write_file / patch_file — внеси изменения ТОЛЬКО в эту ветку.\n"
        "5. diff_vs_base — убедись, что изменения на месте.\n"
        f"6. report_work({issue_number}, '<что сделано>') — оставь комментарий "
        "в issue со ссылкой на ветку.\n\n"
        "Альтернатива шагам 3-6: если после честного изучения issue и кода "
        "правка НЕ требуется — вместо них вызови "
        f"report_no_action_needed({issue_number}, '<конкретная причина>')."
    )

    with kbench.chats.new(f"{model_id}::github_issue_resolution#{issue_number}") as chat:
        kbench.user.send(ISSUE_SYSTEM_PROMPT)
        solved, rounds_used, self_report = run_rounds(
            llm,
            lambda b: make_github_tools(slug, b),
            lambda: check_issue_solved(model_id, slug, issue_number, issue_text),
            task_context,
        )

    ISSUE_ROUNDS_LOG.append({
        "model_id": model_id, "issue": issue_number, "branch": branch,
        "solved": solved, "rounds_used": rounds_used, "self_report": self_report,
    })
    if not solved:
        kbench.assertions.assert_fail(
            expectation=f"issue #{issue_number}: нет ни (ветки {branch} с коммитами "
                        "и комментария со ссылкой на неё), ни подтверждённого судьёй "
                        "report_no_action_needed"
        )
    return solved


kbench.config.enable_console_mode(quiet=False)


## 6. Прогон: каждая модель × каждый открытый issue

In [14]:
MODELS = [
    "anthropic/claude-opus-5@default",
    "google/gemini-3.5-flash-lite",
    "openai/gpt-5.4-nano-2026-03-17",
]

_open_issue_items = [
    it for it in _gh_paginate(f"{REPO_PATH}/issues", params={"state": "open"})
    if "pull_request" not in it
]
# Сортируем по приоритету ДО цикла: если прогон оборвётся на середине списка
# (квота Kaggle Model Proxy, время сессии, rate limit), важные issues уже
# успеют пройти проверку на всех моделях, а не только те, что были первыми
# по номеру.
_open_issue_items.sort(key=lambda it: _issue_priority(it)[0])
open_issues = [it["number"] for it in _open_issue_items]
print("Открытые issues (по приоритету):")
for it in _open_issue_items:
    print(f"  [{_issue_priority(it)[1]}] #{it['number']} {it['title']}")

completed = []
for model_id in MODELS:
    for num in open_issues:
        results = github_issue_resolution_eval.evaluate(
            llm=[kbench.llms[model_id]],
            evaluation_data=pd.DataFrame([{"model_id": model_id, "issue_number": num}]),
            on_failure="continue",
            max_attempts=1,
        )
        df = results.completed_runs.as_dataframe()
        df["model_id"], df["issue"] = model_id, num
        completed.append(df)
        print(f"{model_id} / issue #{num}: "
              f"{len(results.completed_runs)} ok, {len(results.errored_runs)} err")

issue_rounds_df = pd.DataFrame(ISSUE_ROUNDS_LOG)
issue_rounds_df

Открытые issues (по приоритету):
  [??] #10 Нужно изменить, поправить дизайн для использования на телефоне
  [??] #9 Агент не может выполнить команды внутри Codespace
TASK: github_issue_resolution_new: Run #1
  
  [PROMPT]
  Ты — инженер-агент, работающий с реальным GitHub-репозиторием только через
  инструменты. Твой цикл работы: прочитать issue -> изучить релевантные файлы
  (list_files/search_code/read_file) -> создать ветку под issue
  (create_issue_branch) -> внести изменения (write_file/patch_file) -> проверить
  (diff_vs_base) -> отчитаться в issue (report_work). Никогда не выдумывай
  содержимое файлов — сначала читай.

  Дополнительно у тебя есть реальная облачная среда (GitHub Codespaces) поверх
  твоей рабочей ветки — используй её не вместо, а в дополнение к diff_vs_base:
  - start_codespace() — поднять codespace на твоей ветке и получить ссылку,
  которую можно прислать человеку, чтобы он сам открыл и проверил рабочий код;
  - run_in_codespace(command) — самому запустить пр

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:  1.9min finished


  
  [PROMPT]
  Раунд 2/4. Новый бюджет: 10 вызовов.
  Вызовы с прошлого резюме:
  (вызовов не было)
  Не повторяй уже сделанные вызовы без необходимости, продолжай со следующего
  логического шага.
  
  [CHAT: Tool loop]
  
  [PROMPT]
  Раунд 3/4. Новый бюджет: 10 вызовов.
  Вызовы с прошлого резюме:
  (вызовов не было)
  Не повторяй уже сделанные вызовы без необходимости, продолжай со следующего
  логического шага.
  
  [CHAT: Tool loop]
  
  [PROMPT]
  Раунд 4/4. Новый бюджет: 10 вызовов.
  Вызовы с прошлого резюме:
  (вызовов не было)
  Не повторяй уже сделанные вызовы без необходимости, продолжай со следующего
  логического шага.

  ПОСЛЕДНИЙ РАУНД. В конце добавь блок:
  ===SELF_REPORT===
  готово_процентов: <0-100>
  осталось_сделать: <кратко>
  нужно_ещё_шагов: <число>
  ===END_SELF_REPORT===
  
  [CHAT: Tool loop]

--------------------------------------------------------------------------------
  Line  Expectation                                                     Result
----

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    1.8s finished


  
  [RESPONSE: google/gemini-3.5-flash-lite]
  
  [TOOL: get_issue]
  ToolInvocationResult(name='get_issue', arguments={'issue_number': 10}, call_id
  ='call_66109__mpthought__AY89a1/9CRNHuCLOGX8kfYNq81NfXX55igzqRrKpdE4S4HAmEGfKD
  ffXInDrGtVThMFHUYoNkoYolMSeQLauorlsuXz5bHdkHIjdGin+LDDznWs=', output='ISSUE
  #10: Нужно изменить, поправить дизайн для использования на телефоне\nstate:
  open | author: mlaa4ml | labels: -\nurl:
  https://github.com/mlaa4ml/durakProekt/issues/10\n\n--- BODY ---\nСейчас при
  игре в центре стола когда используется разметка для телефона и на столе больше
  трех карт атакующего начинает переезжать сначала блок "бито", а потом и блок
  "колода"? Становится не удобно играть? Нужно найти и предложить в своей ветки
  решение этой проблемы? Кроме того если есть предложения оптимизировать весь
  дизайн, но показать сравнение или оставить возможность увидеть как было и
  стало чтобы потом решить какой вариант выбрать в конечном счёте?', error=None)
  
  [RESPONSE: 

...

  
  [RESPONSE: openai/gpt-5.4-nano-2026-03-17]
  
  [TOOL: get_issue]
  ToolInvocationResult(name='get_issue', arguments={'issue_number': 10},
  call_id='call_UGm2gJrZzVlRyIrdfvZhYpyp', output='ISSUE #10: Нужно изменить,
  поправить дизайн для использования на телефоне\nstate: open | author: mlaa4ml
  | labels: -\nurl: https://github.com/mlaa4ml/durakProekt/issues/10\n\n--- BODY
  ---\nСейчас при игре в центре стола когда используется разметка для телефона и
  на столе больше трех карт атакующего начинает переезжать сначала блок "бито",
  а потом и блок "колода"? Становится не удобно играть? Нужно найти и предложить
  в своей ветки решение этой проблемы? Кроме того если есть предложения
  оптимизировать весь дизайн, но показать сравнение или оставить возможность
  увидеть как было и стало чтобы потом решить какой вариант выбрать в конечном
  счёте?\n\n--- COMMENT by mlaa4ml ---\n### Работа по issue #10\n\nРешена
  проблема с разметкой для мобильных устройств на странице игры (docs/ind

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:  1.5min finished


  
  [RESPONSE: openai/gpt-5.4-nano-2026-03-17]
  
  [TOOL: get_issue]
  ToolInvocationResult(name='get_issue', arguments={'issue_number': 9},
  call_id='call_p9NNjEhiAk7k3Dy1BW0BBNa9', output='ISSUE #9: Агент не может
  выполнить команды внутри Codespace\nstate: open | author: mlaa4ml | labels:
  -\nurl: https://github.com/mlaa4ml/durakProekt/issues/9\n\n--- BODY
  ---\nrun_in_codespace(...) стабильно падает ещё до контейнера, на стороне
  CLI:\n\nexit=1\nstderr: unknown flag: --command\nUsage:  gh codespace ssh
  [<flags>...] [-- <ssh-flags>...] [<command>]\n\nЭто уже не проблема прав
  репозитория, а баг обвязки агента: команда собирается как gh codespace ssh
  --command "...", тогда как современный gh ожидает gh codespace ssh -c <name>
  -- <команда>. Ни node -v, ни npm run smoke внутри Codespace выполнить не
  возможно. Чинится это в инфраструктуре агента, не в этом репозитории, поэтому
  в коде проекта сделано то, что от репозитория зависит, — дал облачную
  проверку, которой пра

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:   55.2s finished


,model_id,issue,branch,solved,rounds_used,self_report
0,anthropic/claude-opus-5@default,10,anthropic-claude-opus-5-default-issue-10,False,4,раунд 4 прерван исключением: PermissionDeniedE...
1,anthropic/claude-opus-5@default,9,anthropic-claude-opus-5-default-issue-9,False,4,раунд 4 прерван исключением: PermissionDeniedE...
2,google/gemini-3.5-flash-lite,10,google-gemini-3-5-flash-lite-issue-10,True,4,раунд 3 прерван исключением: ToolInvocationLim...
3,google/gemini-3.5-flash-lite,9,google-gemini-3-5-flash-lite-issue-9,False,4,готово_процентов: 100\nосталось_сделать: ничег...
4,openai/gpt-5.4-nano-2026-03-17,10,openai-gpt-5-4-nano-2026-03-17-issue-10,False,4,готово_процентов: 20\nосталось_сделать: Довест...
5,openai/gpt-5.4-nano-2026-03-17,9,openai-gpt-5-4-nano-2026-03-17-issue-9,False,4,готово_процентов: 100\nосталось_сделать: нечег...


## 7. Дополнения (был Смоук-тест инструментов без LLM - в самом конце теперь)

Дешёвая проверка, что расширенный слой и scope токена в порядке —
до того, как тратить деньги на модели.

In [15]:
import os
import glob
import zipfile
from datetime import datetime

NOTEBOOK_ROOT = os.path.abspath(os.getcwd())

def archive_run_json(archive_name: str = None) -> str:
    if archive_name is None:
        archive_name = f"github_benchmark_results_{datetime.now():%Y%m%d_%H%M%S}"
    archive_path = os.path.join(NOTEBOOK_ROOT, f"{archive_name}.zip")

    run_json_files = sorted(set(
        glob.glob(os.path.join(NOTEBOOK_ROOT, "*.run.json"))
        + glob.glob(os.path.join(NOTEBOOK_ROOT, "**", "*.run.json"), recursive=True)
    ))

    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in run_json_files:
            zf.write(path, arcname=os.path.basename(path))

    print(f"Архив готов: {archive_path}")
    print(f"  - *.run.json: {len(run_json_files)}")
    return archive_path


archive_path = archive_run_json()

Архив готов: /kaggle/working/github_benchmark_results_20260824_162236.zip
  - *.run.json: 3


In [16]:
import json
import glob
import pandas as pd


def load_github_benchmark_costs(results_dir: str = ".", pattern: str = "*.run.json") -> pd.DataFrame:
    """
    Читает все .run.json из results_dir и строит таблицу с usage/cost по моделям.

    Токены и стоимость берутся из conversations[i]['metrics'] — там, где
    conv['metrics'] непуст, это одна "Tool loop"-беседа = один раунд задачи.
    Стоимость в файлах хранится в нанодолларах (1e9 нанодолларов = $1).
    """
    rows = []

    for path in sorted(glob.glob(f"{results_dir}/{pattern}")):
        with open(path) as f:
            data = json.load(f)

        model_id = data["modelVersion"]["slug"]
        state = data["state"]

        input_tokens = 0
        output_tokens = 0
        input_cost = 0.0
        output_cost = 0.0
        latency_ms = 0
        rounds_used = 0

        for conv in data.get("conversations", []):
            m = conv.get("metrics") or {}
            if not m:
                continue  # пустые metrics — это не раунд, а служебная запись беседы
            rounds_used += 1
            input_tokens += m.get("inputTokens", 0) or 0
            output_tokens += m.get("outputTokens", 0) or 0
            input_cost += int(m.get("inputTokensCostNanodollars", 0) or 0) / 1e9
            output_cost += int(m.get("outputTokensCostNanodollars", 0) or 0) / 1e9
            latency_ms += int(m.get("totalBackendLatencyMs", 0) or 0)

        total_cost = input_cost + output_cost
        cost_per_1m_input = (input_cost / input_tokens * 1e6) if input_tokens else None
        cost_per_1m_output = (output_cost / output_tokens * 1e6) if output_tokens else None

        solved = None
        for r in data.get("results", []):
            if r.get("type") == "AGGREGATED":
                solved = r.get("booleanResult")

        rows.append({
            "model_id": model_id,
            "state": state,
            "solved": solved,
            "rounds_used": rounds_used,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "input_cost_usd": round(input_cost, 6),
            "output_cost_usd": round(output_cost, 6),
            "total_cost_usd": round(total_cost, 6),
            "cost_per_1m_input_usd": round(cost_per_1m_input, 3) if cost_per_1m_input else None,
            "cost_per_1m_output_usd": round(cost_per_1m_output, 3) if cost_per_1m_output else None,
            "latency_ms_total": latency_ms,
        })

    df = pd.DataFrame(rows).sort_values("model_id").reset_index(drop=True)

    # строка TOTAL по всем моделям — через .loc, чтобы не ловить FutureWarning от concat
    # с all-NA колонками (state/solved/cost_per_1m у итоговой строки не определены)
    if not df.empty:
        total_idx = len(df)
        df.loc[total_idx, "model_id"] = "TOTAL (все модели)"
        df.loc[total_idx, "rounds_used"] = df["rounds_used"].sum()
        df.loc[total_idx, "input_tokens"] = df["input_tokens"].sum()
        df.loc[total_idx, "output_tokens"] = df["output_tokens"].sum()
        df.loc[total_idx, "input_cost_usd"] = round(df["input_cost_usd"].sum(), 6)
        df.loc[total_idx, "output_cost_usd"] = round(df["output_cost_usd"].sum(), 6)
        df.loc[total_idx, "total_cost_usd"] = round(df["total_cost_usd"].sum(), 6)
        df.loc[total_idx, "latency_ms_total"] = df["latency_ms_total"].sum()
        # state / solved / cost_per_1m_* для итоговой строки осознанно оставляем NaN

    return df


# использование:
costs_df = load_github_benchmark_costs(results_dir="/kaggle/working", pattern="github_*.run.json")
costs_df

,model_id,state,solved,rounds_used,input_tokens,output_tokens,input_cost_usd,output_cost_usd,total_cost_usd,cost_per_1m_input_usd,cost_per_1m_output_usd,latency_ms_total
0,anthropic/claude-opus-5@default,BENCHMARK_TASK_RUN_STATE_COMPLETED,False,0.0,0.0,0.0,0.000000,0.000000,0.000000,NaN,NaN,0.0
1,google/gemini-3.5-flash-lite,BENCHMARK_TASK_RUN_STATE_COMPLETED,False,4.0,37431.0,1026.0,0.011229,0.002565,0.013794,0.300,2.50,9797.0
2,openai/gpt-5.4-nano-2026-03-17,BENCHMARK_TASK_RUN_STATE_COMPLETED,False,4.0,155483.0,1682.0,0.011351,0.002103,0.013454,0.073,1.25,22119.0
3,TOTAL (все модели),NaN,NaN,8.0,192914.0,2708.0,0.022580,0.004668,0.027248,NaN,NaN,31916.0


In [ ]:
def smoke_test(slug="anthropic-claude-opus-5-default"):
    checks = {
        "branches": gh_list_branches(),
        "files(main)": gh_list_files(GITHUB_BASE_BRANCH)[:300],
        "issues(all)": gh_list_issues("all")[:500],
        "compare": gh_compare(GITHUB_BASE_BRANCH, slug),
        "codespaces(scope)": gh_list_my_codespaces(),
    }
    for k, v in checks.items():
        ok = not (isinstance(v, str) and v.startswith("ERROR:"))
        print(f"[{'OK ' if ok else 'FAIL'}] {k}: {str(v)[:250]}\n")
    print("Если issues -> FAIL 403, у токена нет scope 'Issues: Read and write'.")
    print("Если codespaces -> FAIL, у токена нет scope 'Codespaces: Read and write'.")


smoke_test()